# Frozen research evidence report
This notebook renders the frozen scalar tables. It does not read raw captures, execute experiments, or write thesis conclusions.

Counts, units and unavailable values are retained. Coverage distinguishes implemented analysis from measured evidence and supported comparisons. An available row does not establish that an RQ has been answered.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown, Image, HTML
display(HTML("""<style>
@media print {
  @page { size: A4 landscape; margin: 12mm; }
  .jp-OutputArea-output { overflow: visible !important; }
  table { font-size: 8pt; table-layout: fixed; width: 100%; }
  th, td { white-space: normal !important; overflow-wrap: anywhere; }
  img { max-width: 100% !important; max-height: 175mm; object-fit: contain; }
}
</style>"""))
REPORT = Path.cwd()
metadata = json.loads((REPORT / 'report.json').read_text())
def table(name):
    return pd.read_csv(REPORT / name, dtype=str, keep_default_na=False).replace({r"\N": "unavailable"})
display(Markdown("## Analysis selection"))
for analysis in metadata['analyses']:
    display(Markdown(analysis['id'] + " — baseline: " + analysis['baseline']))
for name in ('dataset-counts.csv', 'coverage.csv', 'cross-domain-status.csv', 'summaries.csv', 'differences.csv', 'outcomes.csv', 'budgets.csv',
             'baremetal_availability.csv', 'ota_outcomes.csv'):
    display(Markdown("## " + name + " — [full CSV](" + name + ")"))
    frame = table(name)
    important = {
        'coverage.csv': ['section','metric','available_summary_rows','supported_comparison_rows','status','limitation'],
        'summaries.csv': ['analysis','condition','metric','value','unit','n_runs','reason'],
        'differences.csv': ['analysis','condition','metric','delta','percent_difference','reason'],
        'outcomes.csv': ['analysis','condition','metric','completed_runs','failed_runs','contributing_runs'],
        'ota_outcomes.csv': ['analysis','scenario','delivery_outcome','transition_outcome','final_identity_reason'],
    }.get(name)
    if important:
        frame = frame[[c for c in important if c in frame.columns]]
    if len(frame) > 60:
        display(Markdown(f"Preview: first 30 of {len(frame)} rows. Download the full CSV for every row and column."))
        frame = frame.head(30)
    display(HTML(frame.to_html(index=False, escape=True)))
display(Markdown("## Figures from the same summary rows"))
for path in sorted(REPORT.glob('figure-*.png')):
    display(Image(filename=str(path)))
display(Markdown("## Interpretation limits"))
for limitation in metadata['limitations']:
    display(Markdown("- " + limitation))
